[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/experiments/agents-cookbook.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# <center>将 Phoenix 与 AI Agent 结合使用</center>

本指南介绍如何使用 Phoenix 创建和评估 Agent，以改进表现。步骤包括：

* 使用路由器模板创建客服 Agent
* 追踪 Agent 活动，包括函数调用
* 创建性能基准测试数据集
* 使用代码、人工标注和 LLM 裁判评估 Agent
* 使用不同提示词和模型进行实验

# 初始设置

配置库、密钥，以及通过 Phoenix 进行的 OpenAI 追踪。

### 安装库

In [ ]:
!pip install -qq "arize-phoenix[evals,llama-index]" "llama-index-llms-openai" "openai>=1" gcsfs nest_asyncio openinference-instrumentation-openai

### 设置密钥

In [ ]:
import os
from getpass import getpass

if not (phoenix_endpoint := os.getenv("PHOENIX_COLLECTOR_ENDPOINT")):
    phoenix_endpoint = getpass("🔑 请输入 Phoenix 收集器端点：")
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = phoenix_endpoint


if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
    phoenix_api_key = getpass("🔑 请输入 Phoenix API 密钥：")
os.environ["PHOENIX_API_KEY"] = phoenix_api_key

if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥：")
os.environ["OPENAI_API_KEY"] = openai_api_key

### 配置追踪

跟随本教程需要注册 Phoenix Cloud，它是我们用于观察 LLM 应用的免费开源方案。参阅[指南](https://arize.com/docs/phoenix/quickstart)，或获取 API 密钥后运行以下代码。

In [ ]:
from phoenix.otel import register

# 配置 Phoenix 追踪器
tracer_provider = register(project_name="agents-cookbook", auto_instrument=True)

# 创建客服 Agent

我们将按以下架构，使用函数调用创建客服 Agent：

<img src="https://storage.cloud.google.com/arize-assets/tutorials/images/agent_architecture.png" width="800"/>

### 配置函数并创建客服 Agent

下面定义 6 个函数：

1. product_comparison
2. product_search
3. customer_support
4. track_package
5. product_details
6. apply_discount_code


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "product_comparison",
            "description": "比较两个商品的特性。",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_a_id": {
                        "type": "string",
                        "description": "商品 A 的唯一标识符。",
                    },
                    "product_b_id": {
                        "type": "string",
                        "description": "商品 B 的唯一标识符。",
                    },
                },
                "required": ["product_a_id", "product_b_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "product_search",
            "description": "根据条件搜索商品。",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "搜索查询字符串。",
                    },
                    "category": {
                        "type": "string",
                        "description": "用于过滤搜索的类别。",
                    },
                    "min_price": {
                        "type": "number",
                        "description": "待搜索商品的最低价格。",
                        "default": 0,
                    },
                    "max_price": {
                        "type": "number",
                        "description": "待搜索商品的最高价格。",
                    },
                    "page": {
                        "type": "integer",
                        "description": "分页页码。",
                        "default": 1,
                    },
                    "page_size": {
                        "type": "integer",
                        "description": "每页结果数量。",
                        "default": 20,
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "customer_support",
            "description": "获取处理问题的客服联系信息。",
            "parameters": {
                "type": "object",
                "properties": {
                    "issue_type": {
                        "type": "string",
                        "description": "问题类型，例如账单或技术支持。",
                    }
                },
                "required": ["issue_type"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "track_package",
            "description": "根据追踪号码查询包裹状态。",
            "parameters": {
                "type": "object",
                "properties": {
                    "tracking_number": {
                        "type": "integer",
                        "description": "包裹追踪号码。",
                    }
                },
                "required": ["tracking_number"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "product_details",
            "description": "返回给定商品 ID 的详情",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_id": {
                        "type": "string",
                        "description": "要查询的商品 ID。",
                    }
                },
                "required": ["product_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_discount_code",
            "description": "将折扣码应用到指定订单。",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "integer",
                        "description": "要应用折扣码的订单 ID。",
                    },
                    "discount_code": {
                        "type": "string",
                        "description": "要应用的折扣码",
                    },
                },
                "required": ["order_id", "discount_code"],
            },
        },
    },
]

下面定义名为 run_prompt 的函数，使用 OpenAI 的聊天补全调用并传入函数。

In [ ]:
import os

import openai


def run_prompt(input):
    client = openai.Client()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        tools=tools,
        tool_choice="auto",
        messages=[
            {
                "role": "system",
                "content": " ",
            },
            {
                "role": "user",
                "content": input,
            },
        ],
    )

    if (
        hasattr(response.choices[0].message, "tool_calls")
        and response.choices[0].message.tool_calls is not None
        and len(response.choices[0].message.tool_calls) > 0
    ):
        tool_calls = response.choices[0].message.tool_calls
    else:
        tool_calls = []

    if response.choices[0].message.content is None:
        response.choices[0].message.content = ""
    if response.choices[0].message.content:
        return response.choices[0].message.content
    else:
        return tool_calls

测试一下是否返回正确函数！将 tool_choice 设置为 "auto" 或 "required" 时，路由器行为会不同。

In [ ]:
run_prompt("Hi, I'd like to apply to apply a discount code to my order.")

基本 Agent 已创建，生成问题数据集并在其上运行提示词！

# 创建合成问题数据集

使用下面的模板生成包含 25 个问题的 DataFrame，用于测试客服 Agent。

In [ ]:
GEN_TEMPLATE = """
你是生成复杂客服问题的助手。
问题应经常包含：

多个类别：逻辑上属于多个类别的问题，例如同时涉及商品详情与折扣码。
模糊细节：信息有限或模糊，需要澄清才能正确分类。
混合意图：客户目标或需求不明确，或在问题内部看似互相冲突。
间接表达：使用间接或礼貌措辞隐藏直接需求，例如“我想知道是否……”或“也许你能帮我……”。

针对具体类别：

包裹追踪：使用“最近”“前一阵子”等模糊时间，而非具体日期。
商品比较与商品搜索：使用“高端智能手机”“节能电器”等一般描述，不给具体商品名或 ID。
应用折扣码：询问折扣是否适用于假设或过去情况，或不说明是否已购买。
商品详情：模糊地询问涉及多个商品或类别的比较或细节，例如“介绍一下适合家庭办公的电子产品系列”。

更有挑战的问题示例：
"There's an issue with one of the items I think I bought last month—what should I do?"
"I need help with something I ordered, or maybe I'm just looking for something new. Can you help?"

部分问题应是对所提供函数的直接使用。

以列表回复，每行一个问题。行首不要编号，不要类别标题。
生成 25 个问题，确保没有重复。
"""

In [ ]:
import nest_asyncio
import pandas as pd

nest_asyncio.apply()
from phoenix.evals import LLM

pd.set_option("display.max_colwidth", 500)

gen_model = LLM(provider="openai", model="gpt-4o")

In [ ]:
resp = gen_model.generate_text(prompt=GEN_TEMPLATE, max_tokens=1300)

In [ ]:
split_response = resp.strip().split("\n")

questions_df = pd.DataFrame(split_response, columns=["question"])
print(questions_df)

现在用此数据集运行上面的路由器提示词！

In [ ]:
response_df = questions_df.copy(deep=True)
response_df["response"] = response_df["question"].apply(run_prompt)
response_df["response"] = response_df["response"].astype(str)

In [ ]:
response_df

# 评估 Agent

有了测试用例，就可以创建评估器衡量表现，无需手动检查每条追踪来确认 LLM 是否正确执行。

这里定义评估模板，判断路由器是否正确选择了函数、所选函数是否合适，以及参数是否填写正确。

In [ ]:
ROUTER_EVAL_TEMPLATE = """你正在比较问题与回复，验证回复是否应调用函数而非直接回答。数据如下：
    [BEGIN DATA]
    ************
    [Question]: {question}
    ************
    [LLM Response]: {response}
    ************
    [END DATA]

将问题与回复比较，判断回复是否决定调用正确函数。
回复必须是单个词，"correct" 或 "incorrect"，不得包含任何其他文字或字符。
"incorrect" 表示 Agent 应调用函数却直接回答，或选择了错误函数。
"correct" 表示所选函数能正确、完整地回答用户问题。

各函数的更多信息：
product_comparison：比较两个商品的特性，应包含商品 ID 或名称。若问题中有名称或 ID，但生成函数中没有，则回复不正确。
product_search：根据条件搜索商品。
track_package：根据追踪号码查询包裹状态。
customer_support：获取处理问题的客服联系信息，回复必须包含邮箱或电话号码。
apply_discount_code：将折扣码应用于订单。
product_details：获取单个商品的详细特性。
"""

FUNCTION_SELECTION_EVAL_TEMPLATE = """你正在比较函数调用回复与问题，判断生成的调用是否正确。数据如下：
    [BEGIN DATA]
    ************
    [Question]: {question}
    ************
    [LLM Response]: {response}
    ************
    [END DATA]

将问题与函数调用比较，判断调用能否返回问题的答案。
重点关注这个具体问题能否由该调用回答。
回复必须是单个词，"correct"、"incorrect" 或 "not-applicable"，不得包含其他文字或字符。
"incorrect" 表示函数调用无法回答问题。
"correct" 表示函数调用一定能够回答问题。
"not-applicable" 表示回复不是函数调用。

各函数的更多信息：
product_comparison：比较两个商品的特性，应包含商品 ID 或名称。若问题中有名称或 ID，但生成函数中没有，则回复不正确。
product_search：根据条件搜索商品。
track_package：根据追踪号码查询包裹状态。
customer_support：获取处理问题的客服联系信息，回复必须包含邮箱或电话号码。
apply_discount_code：将折扣码应用于订单。
product_details：获取单个商品的详细特性。
"""

PARAMETER_EXTRACTION_EVAL_TEMPLATE = """你正在比较函数调用回复与问题，判断生成调用是否从问题中提取了完全正确的参数。数据如下：
    [BEGIN DATA]
    ************
    [Question]: {question}
    ************
    [LLM Response]: {response}
    ************
    [END DATA]

将生成函数的参数与下面的 JSON 比较。
从问题中提取的参数必须与下方 JSON 完全匹配。
回复必须是单个词，"correct"、"incorrect" 或 "not-applicable"，不得包含其他文字或字符。
"incorrect" 表示函数参数不完全符合下方 JSON schema，或生成函数无法正确回答用户问题。
如果回复编造了 JSON schema 中没有的信息，也应回复 "incorrect"。
"correct" 表示函数调用参数符合下方 JSON，且仅提供相关信息。
"not-applicable" 表示回复不是函数调用。

各函数的更多信息：
product_comparison：比较两个商品的特性，应包含商品 ID 或名称。若问题中有名称或 ID，但生成函数中没有，则回复不正确。
product_search：根据条件搜索商品。
track_package：根据追踪号码查询包裹状态。
customer_support：获取处理问题的客服联系信息，回复必须包含邮箱或电话号码。
apply_discount_code：将折扣码应用于订单。
product_details：获取单个商品的详细特性。
"""

使用 Phoenix 的 `ClassificationEvaluator` 和 `async_evaluate_dataframe`，对上面生成的回复 DataFrame 运行评估！

In [ ]:
from phoenix.evals import LLM, ClassificationEvaluator, async_evaluate_dataframe

rails = ["incorrect", "correct"]

router_evaluator = ClassificationEvaluator(
    name="router_eval",
    llm=LLM(provider="openai", model="gpt-4o"),
    prompt_template=ROUTER_EVAL_TEMPLATE,
    choices=rails,
)

function_selection_evaluator = ClassificationEvaluator(
    name="function_selection_eval",
    llm=LLM(provider="openai", model="gpt-4o"),
    prompt_template=FUNCTION_SELECTION_EVAL_TEMPLATE,
    choices=rails,
)

parameter_extraction_evaluator = ClassificationEvaluator(
    name="parameter_extraction_eval",
    llm=LLM(provider="openai", model="gpt-4o"),
    prompt_template=PARAMETER_EXTRACTION_EVAL_TEMPLATE,
    choices=rails,
)

eval_result_df = await async_evaluate_dataframe(
    dataframe=response_df,
    evaluators=[router_evaluator, function_selection_evaluator, parameter_extraction_evaluator],
    concurrency=10,
)

查看并检查评估结果！

In [ ]:
eval_result_df[["question", "response", "router_eval_score", "router_eval_execution_details"]]

In [ ]:
eval_result_df[
    [
        "question",
        "response",
        "function_selection_eval_score",
        "function_selection_eval_execution_details",
    ]
]

In [ ]:
eval_result_df[
    [
        "question",
        "response",
        "parameter_extraction_eval_score",
        "parameter_extraction_eval_execution_details",
    ]
]

# 创建实验

使用上面生成的问题数据集，通过实验功能跟踪 Agent 的模型、提示词和参数变化。

创建此数据集并上传到平台。

In [ ]:
import os
from uuid import uuid1

from phoenix.client import Client

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com"

client = Client()

dataset = client.datasets.create_dataset(
    dataframe=questions_df,
    name="agents-cookbook-" + str(uuid1()),
    input_keys=["question"],
)

In [ ]:
from phoenix.evals import (
    LLM,
    ClassificationEvaluator,
    bind_evaluator,
)

eval_llm = LLM(provider="openai", model="gpt-4o")

routing_eval = bind_evaluator(
    ClassificationEvaluator(
        name="routing",
        llm=eval_llm,
        prompt_template=ROUTER_EVAL_TEMPLATE,
        choices={"correct": 1.0, "incorrect": 0.0},
    ),
    input_mapping={"question": "input.question", "response": "output"},
)

function_call_eval = bind_evaluator(
    ClassificationEvaluator(
        name="function_call",
        llm=eval_llm,
        prompt_template=FUNCTION_SELECTION_EVAL_TEMPLATE,
        choices={"correct": 1.0, "incorrect": 0.0},
    ),
    input_mapping={"question": "input.question", "response": "output"},
)

parameter_extraction_eval = bind_evaluator(
    ClassificationEvaluator(
        name="parameter_extraction",
        llm=eval_llm,
        prompt_template=PARAMETER_EXTRACTION_EVAL_TEMPLATE,
        choices={"correct": 1.0, "incorrect": 0.0},
    ),
    input_mapping={"question": "input.question", "response": "output"},
)

In [ ]:
dataset

In [ ]:
from phoenix.client.experiments import run_experiment


def prompt_gen_task(input):
    return run_prompt(input["question"])


experiment = run_experiment(
    dataset=dataset,
    task=prompt_gen_task,
    evaluators=[routing_eval, function_call_eval, parameter_extraction_eval],
    experiment_name="agents-cookbook",
)